In [13]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import pandas as pd
import rpy2.rinterface_lib.sexp as SEXP
from rpy2.robjects.vectors import ListVector
import os
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.diagnostic import het_breuschpagan
import numpy as np
from scipy.stats import kruskal
from itertools import combinations


In [2]:
parameter_translation_map = {
    "A_viewerServerSelectionDistribution": "SP",
    "B_maxReservation": "MR [B]",
    "B_existingViewerServerSelectionDistribution": "ESP",
    "B_newViewerServerSelectionDistribution": "OSP",
    "B_existingViewerServerSelectionTreeTraversal": "EST",
    "B_newViewerServerSelectionTreeTraversal": "OST",
    "C_maxReservation": "MR [C]",
    "C_reservationType": "RT",
    "C_alreadyLinkedTreeTraversal": "EVST",
    "C_alreadyLinkedDistribution": "EVSP",
    "C_linkedSpaceTreeTraversal": "OVST",
    "C_linkedSpaceDistribution": "OVSP",
    "C_viewerAndLinkFromDistribution": "NVSP",
    "C_publisherServerSelectionDistribution": "PSP",
    "maxReservation": "MR",
}
distribution_cols = [
    "SP", "ESP", "OSP", "EVSP", "OVSP", "NVSP", "PSP"
]
tree_traversal_cols = [
    "EST", "OST", "EVST", "OVST"
]
distribution_translation_map = {
    "LOW_LOAD": "LL",
    "HIGH_LOAD": "HL",
    "ROUND_ROBIN": "RR",
    "RANDOM": "RND",
}
tree_traversal_translation_map = {
    "TOP_TO_BOTTOM": "TTB",
    "BOTTOM_TO_TOP": "BTT",
    "IGNORE_TREE_LEVEL": "IT",
}
reservation_type_translation_map = {
    "RESERVATION_PER_SERVER": "SER",
    "RESERVATION_PER_SESSION": "SES",
}
alg_a_parameters = ["SP"]
alg_b_parameters = ["MR [B]", "ESP", "OSP", "EST", "OST", "MR"]
alg_c_parameters = [
    "MR [C]", "RT", "EVST", "EVSP", "OVST", "OVSP", "NVSP", "PSP", "MR"
]

In [3]:
def load_irace_elite_configs(file_location):
    ro.r['load'](file_location)
    objects = ro.r['iraceResults']
    testing = ListVector(objects.rx2("testing"))
    test_exp_matrix = testing.rx2("experiments")
    test_matrix_row_names = list(test_exp_matrix.rownames)
    test_matrix_col_names = list(test_exp_matrix.colnames)
    training_exps = objects.rx2("experiments")
    training_exps_row_names = list(training_exps.rownames)
    training_exps_col_names = list(training_exps.colnames)
    with (ro.default_converter + pandas2ri.converter).context():
        test_exp_matrix = pandas2ri.rpy2py(test_exp_matrix)
        test_exp_matrix = pd.DataFrame(test_exp_matrix, index=test_matrix_row_names, columns=test_matrix_col_names)
        training_exps = pandas2ri.rpy2py(training_exps)
        training_exps = pd.DataFrame(training_exps, index=training_exps_row_names, columns=training_exps_col_names)
        configs = objects.rx2('allConfigurations')
        configs = configs.replace(-2147483648, None)
        for col in configs.columns:
            configs[col] = configs[col].apply(lambda x: None if isinstance(x, SEXP.NACharacterType) else x)
        configs = configs.drop(columns=['.ID.', '.PARENT.'])
        configs = configs.rename(columns=parameter_translation_map)
        for col in configs.select_dtypes(include=['float']):
            if (configs[col] % 1 == 0).all():
                configs[col] = configs[col].astype('Int64')
        for col in configs.columns:
            if col in distribution_cols :
                configs[col] = configs[col].map(distribution_translation_map)
            if col in tree_traversal_cols:
                configs[col] = configs[col].map(tree_traversal_translation_map)
            if col == "RT":
                configs[col] = configs[col].map(reservation_type_translation_map)
        elites = objects.rx2('allElites')[-1]
        elite_configs = configs.iloc[elites-1]
        elite_configs = elite_configs.dropna(axis=1, how='all')
        for col in elite_configs.select_dtypes(include=['float']):
            if (elite_configs[col] % 1 == 0).all():
                elite_configs[col] = elite_configs[col].astype('Int64')
        all_exps = pd.concat([training_exps, test_exp_matrix], axis=0)
        return configs, elite_configs, test_exp_matrix, training_exps, all_exps

configs_map = {}
elite_configs_map = {}
testing_exps_map = {}
training_exps_map = {}
full_experiments_map = {}
for root, _, files in os.walk("irace_results"):
    for file in files:
        if file.endswith("irace.Rdata"):
            file_path = os.path.join(root, file)
            dir_path = os.path.dirname(file_path)
            splitted = dir_path.split(os.sep)
            budget = splitted[1].split('_')[0]
            instance_type = splitted[2]
            server_capacity = splitted[3]
            try:
                configs, elite_configs, testing_exps, full_exps, all_exps = load_irace_elite_configs(file_path)
                # Ensure 'small' comes before 'medium' in the map
                if instance_type not in configs_map:
                    configs_map[instance_type] = {}
                configs_map[instance_type][server_capacity] = configs
                if instance_type not in elite_configs_map:
                    elite_configs_map[instance_type] = {}
                elite_configs_map[instance_type][server_capacity] = elite_configs
                if instance_type not in testing_exps_map:
                    testing_exps_map[instance_type] = {}
                testing_exps_map[instance_type][server_capacity] = testing_exps
                if instance_type not in training_exps_map:
                    training_exps_map[instance_type] = {}
                training_exps_map[instance_type][server_capacity] = full_exps
                if instance_type not in full_experiments_map:
                    full_experiments_map[instance_type] = {}
                full_experiments_map[instance_type][server_capacity] = all_exps
            except Exception as e:
                print(f"Failed to process {file_path}: {e}")

# Reorder the maps so 'small' comes first, then 'medium', and server capacities are sorted numerically
def reorder_instance_types_and_capacities(d):
    ordered = {}
    for instance in ['small', 'medium']:
        if instance in d:
            # Sort server capacities numerically
            capacities = d[instance]
            ordered_capacities = dict(sorted(capacities.items(), key=lambda x: int(x[0])))
            ordered[instance] = ordered_capacities
    for instance in d:
        if instance not in ordered:
            capacities = d[instance]
            ordered_capacities = dict(sorted(capacities.items(), key=lambda x: int(x[0])))
            ordered[instance] = ordered_capacities
    return ordered

configs_map = reorder_instance_types_and_capacities(configs_map)
elite_configs_map = reorder_instance_types_and_capacities(elite_configs_map)
testing_exps_map = reorder_instance_types_and_capacities(testing_exps_map)
training_exps_map = reorder_instance_types_and_capacities(training_exps_map)
full_experiments_map = reorder_instance_types_and_capacities(full_experiments_map)

In [ ]:
wilcoxon_tests = pd.DataFrame(columns=["Instance Type", "Server Capacity", "Algorithm 1", "Algorithm 2" "W", "p-value"])
for instance_type, server_capacities in full_experiments_map.items():
    for server_capacity, full_exps in server_capacities.items():
        configs = configs_map[instance_type][server_capacity]

        algo_ids = {
            algo: configs[configs["algorithm"] == algo].index.astype(str).tolist()
            for algo in configs["algorithm"].unique()
        }

        results = []

        for algo1, algo2 in combinations(algo_ids.keys(), 2):
            ids1 = algo_ids[algo1]
            ids2 = algo_ids[algo2]

            if not ids1 or not ids2:
                continue

            valid_rows = full_exps[ids1 + ids2]

            # TODO: Find pairs with both values not None and prepare pairs for Wilcoxon test

KeyError: "None of [Index([                   ('1',),                    ('4',),\n                          ('6',),                    ('8',),\n                      ('1', '2'),                ('1', '6'),\n                      ('1', '8'),                ('2', '0'),\n                      ('2', '1'),                ('2', '4'),\n       ...\n       ('1', '3', '0', '6', '2'), ('1', '3', '0', '6', '5'),\n       ('1', '3', '0', '7', '9'), ('1', '3', '0', '8', '7'),\n       ('1', '3', '0', '9', '5'), ('1', '3', '1', '2', '2'),\n       ('1', '3', '1', '3', '0'), ('1', '3', '1', '6', '1'),\n       ('1', '3', '1', '7', '1'), ('1', '3', '1', '7', '6')],\n      dtype='object', length=11800)] are in the [index]"

In [ ]:
merged_map = {}


def calc_deviations(instance_type, server_capacity, merged):
    deviation_df = merged.copy()
    deviation_df["min_objective_per_instance"] = deviation_df.groupby("instance")[
        "objective"
    ].transform("min")
    # Use percentage deviation
    deviation_df["Deviation to best (%)"] = (
        (deviation_df["objective"] - deviation_df["min_objective_per_instance"])
        / deviation_df["min_objective_per_instance"]
    ) * 100

    mean_deviation_per_instance = (
        deviation_df.groupby("instance")["Deviation to best (%)"].mean().sort_index()
    )

    elite_configs = elite_configs_map[instance_type][server_capacity]
    elite_merged = merged[merged["config_id"].isin(elite_configs.index.astype(str))]
    elite_deviation_df = elite_merged.copy()
    elite_deviation_df["min_objective_per_instance"] = deviation_df.groupby("instance")[
        "objective"
    ].transform("min")
    elite_deviation_df["Deviation to best (%)"] = (
        (
            elite_deviation_df["objective"]
            - elite_deviation_df["min_objective_per_instance"]
        )
        / elite_deviation_df["min_objective_per_instance"]
    ) * 100
    return mean_deviation_per_instance, elite_deviation_df


def plot_deviation_from_best(
    mean_deviation,
    elite_deviation,
    only_elite=False,
    ax=None,
):
    plt.figure(figsize=(24, 10))
    if ax is None:
        fig, ax = plt.subplots(figsize=(24, 10))
    elite_config_ids = elite_deviation["config_id"].unique()
    palette = sns.color_palette("husl", len(elite_config_ids))
    width = 0.8 / (len(elite_config_ids) + 1) if len(elite_config_ids) > 0 else 0.8

    x = np.arange(len(mean_deviation.index))

    if not only_elite:
        # Plot the mean deviation to best for all configs as the base bar (light color)
        sns.barplot(
            x=mean_deviation.index.astype(str),
            y=mean_deviation.values,
            color="C0",
            label="Mean Deviation (%) (All Configs)",
            ax=ax,
        )

    # Overlay bars for each elite config_id, using different colors
    for i, config_id in enumerate(elite_config_ids):
        df = elite_deviation[elite_deviation["config_id"] == config_id]
        values = (
            df.groupby("instance")["Deviation to best (%)"]
            .mean()
            .reindex(mean_deviation.index)
        )
        # Offset the bars so they don't overlap
        ax.bar(
            x + (i - len(elite_config_ids) / 2) * width + width / 2,
            values.values,
            width=width,
            color=palette[i],
            label=f"Elite config {i+1}",
        )

    ax.set_xticks(np.arange(len(mean_deviation.index)))
    ax.set_xticklabels(mean_deviation.index.astype(str))
    ax.set_title(
        f"Mean Deviation to Best Solution per Instance (%)\n({instance_type}, Server Capacity {server_capacity})"
    )
    ax.set_xlabel("Instance")
    ax.set_ylabel("Mean Deviation to Best (%)")
    ax.set_ylim(0, 100)
    ax.legend()


def summarize_deviations(
    mean_deviation_all,
    elite_deviation_all,
    mean_deviation_training,
    elite_deviation_training,
    mean_deviation_test,
    elite_deviation_test,
):
    # Calculate summary statistics for mean deviation per instance (all configs)
    summary_df = pd.DataFrame(
        {
            "Avg Mean Deviation (%) (All)": [mean_deviation_all.mean()],
            "Min Mean (All)": [mean_deviation_all.min()],
            "Max Mean (All)": [mean_deviation_all.max()],
            "Std Mean (All)": [mean_deviation_all.std()],
            "Avg Mean Deviation (%) (Training)": [mean_deviation_training.mean()],
            "Min Mean (Training)": [mean_deviation_training.min()],
            "Max Mean (Training)": [mean_deviation_training.max()],
            "Std Mean (Training)": [mean_deviation_training.std()],
            # "Avg Mean Deviation (%) (Testing)": [mean_deviation_test.mean()],
            # "Min Mean (Testing)": [mean_deviation_test.min()],
            # "Max Mean (Testing)": [mean_deviation_test.max()],
            # "Std Mean (Testing)": [mean_deviation_test.std()],
        }
    )
    summary_df.index = ["All Configs"]

    elite_config_ids = elite_deviation_all["config_id"].unique()
    # Calculate summary statistics for each elite config individually
    elite_config_stats = []
    for i in range(len(elite_config_ids)):
        config_id = elite_config_ids[i]
        df_all = elite_deviation_all[elite_deviation_all["config_id"] == config_id]
        all_per_instance = (
            df_all.groupby("instance")["Deviation to best (%)"].mean().sort_index()
        )
        df_training = elite_deviation_training[
            elite_deviation_training["config_id"] == config_id
        ]
        training_per_instance = (
            df_training.groupby("instance")["Deviation to best (%)"].mean().sort_index()
        )
        df_test = elite_deviation_test[elite_deviation_test["config_id"] == config_id]
        test_per_instance = (
            df_test.groupby("instance")["Deviation to best (%)"].mean().sort_index()
        )
        elite_config_stats.append(
            {
                "config_id": "Elite Config " + str(i + 1),
                "Avg Mean Deviation (%) (All)": all_per_instance.mean(),
                "Min Mean (All)": all_per_instance.min(),
                "Max Mean (All)": all_per_instance.max(),
                "Std Mean (All)": all_per_instance.std(),
                "Avg Mean Deviation (%) (Training)": training_per_instance.mean(),
                "Min Mean (Training)": training_per_instance.min(),
                "Max Mean (Training)": training_per_instance.max(),
                "Std Mean (Training)": training_per_instance.std(),
                # "Avg Mean Deviation (%) (Testing)": test_per_instance.mean(),
                # "Min Mean (Testing)": test_per_instance.min(),
                # "Max Mean (Testing)": test_per_instance.max(),
                # "Std Mean (Testing)": test_per_instance.std(),
            }
        )
    if elite_config_stats:
        elite_config_stats_df = pd.DataFrame(elite_config_stats).set_index("config_id")
        elite_config_stats_df.index.name = "Elite Config ID"
    # Round and display all together
    display(
        f"Mean Deviation (%) Summary Statistics for instance type {instance_type} and server capacity {server_capacity}:"
    )
    display(pd.concat([summary_df, elite_config_stats_df]).round(3))


for instance_type, server_capacities in full_experiments_map.items():
    for server_capacity, full_exps in server_capacities.items():
        configs = configs_map[instance_type][server_capacity]
        # only keep the columns that are in configs
        c_configs = configs[configs["algorithm"] == "C"].dropna(axis=1, how="all")
        c_configs = c_configs.drop(columns=["algorithm"]).rename(
            columns={"MR [C]": "MR"}
        )
        e = full_exps[c_configs.index]

        # Prepare data for ANOVA: melt experiments to long format and merge with parameter values
        exp_long = e.reset_index().melt(
            id_vars="index", var_name="config_id", value_name="objective"
        )
        exp_long = exp_long.rename(columns={"index": "instance"})
        exp_long["config_id"] = exp_long["config_id"].astype(str)
        c_configs_ = c_configs.copy()
        c_configs_["config_id"] = c_configs_.index.astype(str)
        merged = exp_long.merge(c_configs_, on="config_id")
        merged["MR"] = merged["MR"].astype(int)

        merged_testing = merged[merged["instance"].str.endswith("t")]
        merged_training = merged[~merged["instance"].str.endswith("t")]

        mean_deviation_per_instance, elite_deviation_df = calc_deviations(
            instance_type, server_capacity, merged
        )
        mean_deviation_per_instance_training, elite_deviation_training_df = (
            calc_deviations(instance_type, server_capacity, merged_training)
        )
        mean_deviation_per_instance_test, elite_deviation_test_df = calc_deviations(
            instance_type, server_capacity, merged_testing
        )

        summarize_deviations(
            mean_deviation_per_instance,
            elite_deviation_df,
            mean_deviation_per_instance_training,
            elite_deviation_training_df,
            mean_deviation_per_instance_test,
            elite_deviation_test_df,
        )

        # plot_deviation_from_best(
        #     mean_deviation_per_instance_training, elite_deviation_training_df
        # )

        if instance_type not in merged_map:
            merged_map[instance_type] = {}
        merged_map[instance_type][server_capacity] = merged

In [ ]:
# Collect Kruskal-Wallis results for all instance types and server capacities into a single DataFrame
all_kruskal_results = []

for instance_type, server_capacities in training_exps_map.items():
    for server_capacity, full_exps in server_capacities.items():
        instance_capacity_kruskal_results = []
        merged = merged_map[instance_type][server_capacity]
        df_clean = merged.dropna(subset=['objective'])
        param_cols = df_clean.columns.difference(['instance', 'config_id', 'objective'])
        params_for_analysis = list(param_cols)
        for param in params_for_analysis:
            groups = [group['objective'].values for name, group in df_clean.groupby(param, observed=False)]
            if len(groups) > 1:
                stat, p = kruskal(*groups)
                result = {
                    "Instance Type": instance_type,
                    "Server Capacity": server_capacity,
                    "Parameter": param,
                    "H-statistic": stat,
                    "p-value": p
                }
                all_kruskal_results.append(result)
                instance_capacity_kruskal_results.append(result)
        # Display sorted DataFrame for this instance_type
        if instance_capacity_kruskal_results:
            df_instance = pd.DataFrame(instance_capacity_kruskal_results)
            df_instance = df_instance.sort_values("H-statistic", ascending=False)
            df_instance = df_instance.drop(columns=["Instance Type", "Server Capacity"])
            display(f"Kruskal-Wallis results for {instance_type}, server capacity {server_capacity}:")
            display(df_instance.style.hide(axis="index"))

all_kruskal_df = pd.DataFrame(all_kruskal_results)
# sort by parameter
all_kruskal_df = all_kruskal_df.sort_values(
    by=['Parameter', 'Instance Type', 'Server Capacity'],
    ascending=[True, False, True],
    key=lambda col: col if col.name != 'Server Capacity' else col.astype(int)
)
display(all_kruskal_df.style.hide(axis="index"))



In [ ]:
for instance_type, server_capacities in training_exps_map.items():
    for server_capacity, full_exps in server_capacities.items():
        merged = merged_map[instance_type][server_capacity]
        df_clean = merged.dropna(subset=['objective']).copy()

        # Get elite config ids for this instance/capacity
        elite_configs = elite_configs_map[instance_type][server_capacity]
        elite_configs = elite_configs.rename(columns={"MR [C]": "MR"})
        elite_ids = set(elite_configs.index.astype(str))

        # Compute deviation from the best for each instance
        df_clean.loc[:, "min_objective_per_instance"] = df_clean.groupby("instance")["objective"].transform("min")
        df_clean.loc[:, "Mean Deviation to best (%)"] = (
            (df_clean["objective"] - df_clean["min_objective_per_instance"]) / df_clean["min_objective_per_instance"]
        ) * 100

        # Compute and display mean deviation from best for each parameter in the data
        param_cols = df_clean.columns.difference(['instance', 'config_id', 'objective', 'min_objective_per_instance', 'Mean Deviation to best (%)'])
        for param in param_cols:
            mean_deviation_param = df_clean.groupby(param)["Mean Deviation to best (%)"].mean().sort_values()

            # For each parameter value, find elite config ids that chose it
            elite_param_map = {}
            for value in mean_deviation_param.index:
                elite_ids_with_value = elite_configs[elite_configs[param] == value].index.astype(str).tolist()
                elite_param_map[value] = elite_ids_with_value if elite_ids_with_value else None

            # Create a DataFrame for display
            result_df = mean_deviation_param.round(3).to_frame("Mean Deviation to best (%)")
            result_df["In Elite"] = result_df.index.map(lambda v: elite_param_map[v])

            display(f"Mean Deviation to Best (%) for {instance_type}, Server Capacity {server_capacity} by {param}:")
            display(result_df)


In [ ]:
# Scatterplot: MR vs objective, colored by RT
for instance_type, server_capacities in training_exps_map.items():
    for server_capacity, full_exps in server_capacities.items():
        merged = merged_map[instance_type][server_capacity]
        for col in merged.columns:
            if col not in ["instance", "objective", "config_id"]:
                fig, axs = plt.subplots(figsize=(24, 30), ncols=3, nrows=10)
                unique_instances = merged["instance"].unique()
                axs_flat = axs.flatten()
                fig.suptitle(f"Scatterplot of {col} vs Objective", y=0.98)
                for i, instance in enumerate(unique_instances):
                    ax = axs_flat[i]
                    sns.scatterplot(
                        data=merged[merged["instance"] == instance],
                        x=col,
                        y="objective",
                        ax=ax
                    )
                    ax.set_title(
                        f"Instance {instance} ({instance_type}, Server Capacity {server_capacity})"
                    )
                    ax.set_xlabel(col)
                    ax.set_ylabel("Objective")
                # Hide unused axes
                for j in range(len(unique_instances), len(axs_flat)):
                    axs_flat[j].set_visible(False)
                plt.tight_layout(rect=[0, 0, 1, 0.96])
                plt.subplots_adjust(top=0.93)
        break


In [ ]:

# testing_exp is a DataFrame where each column represents a parameter configuration, each row an instance.
# The cell contains the value of the objective function for that configuration ran on that instance.
def calculate_statistics(testing_exp):
    # Calculate mean, std, and 95% CI for each configuration (column), taking into account all instances (rows)
    means = testing_exp.mean(axis=0)
    stds = testing_exp.std(axis=0, ddof=1)
    n = testing_exp.shape[0]
    ci95 = 1.96 * stds / (n ** 0.5)

    stats = pd.DataFrame({
        "Mean": means,
        "Std Dev": stds,
        "95% CI": ci95
    })

    # Sort by mean (assuming lower is better; change to ascending=False if higher is better)
    stats = stats.sort_values("Mean")
    return stats


for instance_type, server_capacities in testing_exps_map.items():
    for server_capacity, test_exp in server_capacities.items():
        stats = calculate_statistics(test_exp)
        display(f"Instance Type: {instance_type}, Server Capacity: {server_capacity}")
        display(stats)

In [ ]:
# calculate chosen parameter frequency
def calculate_chosen_parameter_frequency(configs):
    frequency = {}
    for col in configs.columns:
        if col == "algorithm" or col in alg_c_parameters:
            if col not in frequency:
                frequency[col] = {}
            for value in configs[col]:
                if value is None or (isinstance(value, float) and pd.isna(value)):
                    continue
                if isinstance(value, float) and value.is_integer():
                    value = int(value)
                if value not in frequency[col]:
                    frequency[col][value] = 0
                frequency[col][value] += 1
    return frequency


# for every instance type and server capacity, calculate the chosen parameter frequency
chosen_parameter_frequency_map = {}
for instance_type, server_capacities in configs_map.items():
    for server_capacity, config in server_capacities.items():
        if instance_type not in chosen_parameter_frequency_map:
            chosen_parameter_frequency_map[instance_type] = {}
        frequency = calculate_chosen_parameter_frequency(config)
        chosen_parameter_frequency_map[instance_type][server_capacity] = frequency
# for every instance type and server capacity, display the chosen parameter frequency
# Use subplots for every instance type and server capacity
for instance_type, server_capacities in chosen_parameter_frequency_map.items():
    for server_capacity, frequency in server_capacities.items():
        all_params = ["algorithm"] + sorted([p for p in frequency.keys() if p != "algorithm"])
        n_params = len(all_params)
        ncols = 3
        nrows = (n_params - 1 + ncols) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows))
        fig.suptitle(
            f"Parameter Frequency for {instance_type} - Server capacity {server_capacity}\n(Only parameters used by C algorithm are shown)",
        )
        axes = axes.flatten()
        for idx, param in enumerate(all_params):
            param_values = frequency[param]
            if param_values:
                freq_series = pd.Series(param_values)
                # Check if all keys are ints (or can be safely cast to int)
                try:
                    keys_as_int = [int(k) for k in freq_series.index]
                    is_all_int = all(
                        isinstance(k, int)
                        or (isinstance(k, float) and k.is_integer())
                        for k in freq_series.index
                    )
                except Exception:
                    is_all_int = False
                    freq_series = freq_series.sort_index()
                if is_all_int and len(freq_series) > 1:
                    # Expand the series for histogram plotting
                    expanded = []
                    for k, v in freq_series.items():
                        expanded.extend([int(k)] * v)
                    axes[idx].hist(expanded, bins=20, edgecolor="black")
                    axes[idx].set_title(f"{param} histogram")
                    axes[idx].set_ylabel("Frequency")
                else:
                    sns.barplot(
                        x=freq_series.index.astype(str),
                        y=freq_series.values,
                        ax=axes[idx],
                        edgecolor="black"
                    )
                    axes[idx].set_title(f"{param} frequency")
                    axes[idx].set_xlabel("Parameter Value")
                    axes[idx].set_ylabel("Frequency")
            else:
                axes[idx].set_visible(False)
        for idx in range(len(all_params), len(axes)):
            axes[idx].set_visible(False)
        plt.tight_layout(rect=[0, 0, 1, 0.96])

In [ ]:
#heatmap of server capacity and max reservation in elite configs
heatmap_data = []
for instance_type, server_capacities in elite_configs_map.items():
    for server_capacity, elite_config in server_capacities.items():
        if "MR [C]" in elite_config.columns:
            heatmap_data.append({
                "Instance Type": instance_type,
                "Server Capacity": server_capacity,
                "Max Reservation": elite_config[elite_config["RT"] == "SER"]["MR [C]"].mean()
            })
        if "MR" in elite_config.columns:
            heatmap_data.append({
                "Instance Type": instance_type,
                "Server Capacity": server_capacity,
                "Max Reservation": elite_config[elite_config["RT"] == "SER"]["MR"].mean()
            })

heatmap_df = pd.DataFrame(heatmap_data)
heatmap_pivot = heatmap_df.pivot(index="Instance Type", columns="Server Capacity", values="Max Reservation")

# Sort columns numerically for correct order in heatmap
sorted_columns = sorted(heatmap_pivot.columns, key=lambda x: int(x))
heatmap_pivot = heatmap_pivot[sorted_columns]

plt.figure(figsize=(8, 4))
sns.heatmap(heatmap_pivot, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Heatmap of Average Max Reservation by Instance Type and Server Capacity\non Elite Configurations (Only Reservation Per Server)")
plt.ylabel("Instance Type")
plt.xlabel("Server Capacity")